# Lab2：RNN/LSTM 姓名分类实验

**姓名：谢小珂**  
**学号：2310422**

本 notebook 用于课堂提交检查，完整工程代码见 `name_rnn_lstm_experiment/`，实验报告见 `2310422-谢小珂-RNN实验报告.pdf`。


## 1. 实验目标

基于 PyTorch 实现字符级姓名分类任务，对比基础 RNN 与 LSTM 在同一数据集上的分类效果。实验包含：

1. 读取并预处理 `data/names/` 中的姓名数据；
2. 将姓名转换为字符级 one-hot 序列；
3. 构建 RNN 与 LSTM 分类模型；
4. 训练、验证并保存指标；
5. 绘制 loss、accuracy、macro-F1、混淆矩阵等结果图。

In [ ]:
from pathlib import Path
import sys
import pandas as pd
from IPython.display import display, Image

PROJECT_DIR = Path("name_rnn_lstm_experiment").resolve()
SRC_DIR = PROJECT_DIR / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

print("Project directory:", PROJECT_DIR)
print("Source directory:", SRC_DIR)


## 2. 数据集概况

实验使用 PyTorch 字符级姓名分类教程中的 names 数据集。数据处理结果如下：

| split      |   n_samples |   n_classes |   n_letters |   mean_length |
|:-----------|------------:|------------:|------------:|--------------:|
| all        |       20074 |          18 |          58 |          7.15 |
| train      |       17062 |          18 |          58 |          7.16 |
| validation |        3012 |          18 |          58 |          7.12 |


In [ ]:
from name_rnn_lstm_experiment.data_utils import load_name_records, dataset_summary, N_LETTERS

records, labels = load_name_records(PROJECT_DIR / "data" / "names")
summary = dataset_summary(records, labels)

print("Number of samples:", summary["n_samples"])
print("Number of classes:", summary["n_classes"])
print("Number of input letters:", N_LETTERS)
print("Labels:", labels)


## 3. 模型结构

本实验实现并对比两个模型：

- `CharRNN`：使用 `nn.RNN` 作为基础循环网络；
- `CharLSTM`：使用 `nn.LSTM` 作为改进模型，并加入 dropout 与线性分类层。

模型源码位于：

```text
name_rnn_lstm_experiment/src/name_rnn_lstm_experiment/models.py
```

In [ ]:
from name_rnn_lstm_experiment.models import CharRNN, CharLSTM

rnn = CharRNN(input_size=N_LETTERS, hidden_size=128, output_size=len(labels))
lstm = CharLSTM(input_size=N_LETTERS, hidden_size=128, output_size=len(labels), num_layers=1, dropout=0.2)

print("RNN model:")
print(rnn)
print("\nLSTM model:")
print(lstm)


## 4. 训练方法

完整训练脚本为：

```text
name_rnn_lstm_experiment/run_experiment.py
```

为了方便老师检查，根目录补充了 `Lab2.py` 作为统一入口。

In [ ]:
# 快速测试能否跑通训练流程：
# !python Lab2.py --smoke-test --device cpu

# 重新完整训练：
# !python Lab2.py --epochs 50 --device auto


## 5. 实验结果

已生成的主要指标如下：

| model   |   best_epoch |   epochs_ran |   val_loss |   accuracy |   macro_precision |   macro_recall |   macro_f1 |
|:--------|-------------:|-------------:|-----------:|-----------:|------------------:|---------------:|-----------:|
| rnn     |           21 |           31 |     0.6423 |     0.8031 |            0.5491 |         0.477  |     0.501  |
| lstm    |           22 |           32 |     0.5862 |     0.8207 |            0.5645 |         0.4508 |     0.4774 |


In [ ]:
metrics = pd.read_csv(PROJECT_DIR / "results" / "outputs" / "metrics_summary.csv")
display(metrics)


### Loss 曲线

![Loss 曲线](name_rnn_lstm_experiment/results/figures/loss_curve.png)

### Accuracy 曲线

![Accuracy 曲线](name_rnn_lstm_experiment/results/figures/accuracy_curve.png)

### Macro-F1 曲线

![Macro-F1 曲线](name_rnn_lstm_experiment/results/figures/macro_f1_curve.png)

### 模型对比

![模型对比](name_rnn_lstm_experiment/results/figures/model_comparison.png)

### RNN 混淆矩阵

![RNN 混淆矩阵](name_rnn_lstm_experiment/results/figures/rnn_confusion_matrix.png)

### LSTM 混淆矩阵

![LSTM 混淆矩阵](name_rnn_lstm_experiment/results/figures/lstm_confusion_matrix.png)

## 6. 实验结论

从验证集准确率看，LSTM 的结果优于基础 RNN；从 loss 曲线与混淆矩阵看，LSTM 对序列信息的保留能力更强，能够缓解普通 RNN 在较长姓名序列中信息衰减的问题。

本提交同时保留完整源码、数据、训练结果和报告，便于复现实验和课堂检查。